# AI-Powered Visual Defect Detection System

## Deployment using Streamlit

## Objective

The objective of this notebook is to deploy the trained ResNet18 model as an interactive application using Streamlit. Users can upload an image of a surface defect and receive the predicted defect category.

## Import Required Libraries

In [1]:
import torch
import torchvision.transforms as transforms

from torchvision.models import resnet18
from torch import nn

from PIL import Image

## Load Trained Model

In [3]:
model = resnet18(weights=None)

model.fc = nn.Linear(512, 6)

model.load_state_dict(
    torch.load("../models/resnet18_final.pth")
)

model.eval()

print("Model Loaded Successfully")

Model Loaded Successfully


## Define Defect Classes

In [4]:
classes = [
    "crazing",
    "inclusion",
    "patches",
    "pitted_surface",
    "rolled-in_scale",
    "scratches"
]

## Image Preprocessing Pipeline

In [6]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

## Prediction Function

In [7]:
def predict_image(image):

    image = transform(image)

    image = image.unsqueeze(0)

    with torch.no_grad():

        outputs = model(image)

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        confidence, predicted = torch.max(
            probabilities,
            1
        )

    return (
        classes[predicted.item()],
        confidence.item() * 100
    )

## Testing the Prediction Function

In [10]:
import os
import random

class_name = random.choice(classes)

folder = os.path.join(
    "../Dataset/validation/images",
    class_name
)

file_name = random.choice(
    os.listdir(folder)
)

test_image = Image.open(
    os.path.join(folder, file_name)
)

prediction, confidence = predict_image(
    test_image
)

print("Actual Class:", class_name)
print("Predicted Class:", prediction)
print(f"Confidence: {confidence:.2f}%")

Actual Class: rolled-in_scale
Predicted Class: rolled-in_scale
Confidence: 98.50%


## Deployment Summary

The trained ResNet18 model was successfully loaded and tested for prediction. A preprocessing pipeline was implemented to prepare uploaded images for inference. The model is now ready to be integrated into a Streamlit web application for real-time visual defect detection.